In [ ]:
import xarray as xr

import eradiate
from eradiate.data.convert import aer_v1_to_aer_core_v2

eradiate.set_mode("mono")

In [ ]:
ds = eradiate.fresolver.load_dataset("aerosol/govaerts_2021-desert-extrapolated.nc")
# ds["phase"] = ds["phase"].assign_attrs(units="1 / sr")
display(ds)
ds = aer_v1_to_aer_core_v2(ds)
display(ds)

In [ ]:
from eradiate.radprops._particles import ParticleProperties
from eradiate.units import unit_registry as ureg

props = ParticleProperties(ds)
print(f"{props.eval_ssa(550. * ureg.nm) = }")
print(f"{props.eval_ext(550. * ureg.nm) = }")
print(f"{props.eval_phase(550. * ureg.nm) = }")

In [ ]:
import numpy as np
from axsdb.math import interp1d

from eradiate.data.convert import make_aer_core_v2


def _convert_units(value: str):
    if value == "per cent":
        return "percent"
    return value


def _get_units(ds, var, fallback_units=None):
    if "units" in ds[var].attrs:
        units = ds[var].attrs["units"]
        if units == "-":
            units = ""
        return ureg.Unit(_convert_units(units))
    elif fallback_units is not None and var in fallback_units:
        return ureg.Unit(_convert_units(fallback_units[var]))
    else:
        raise ValueError(
            "load_aerosol_libradtran(): The input dataset specifies no units "
            f"for variable '{var}'; this can be addressed by passing them through "
            "the 'fallback_units' parameter."
        )


def libradtran_to_aer_core_v2(
    ds: xr.Dataset,
    fallback_units: dict[str, str] | None = None,
) -> xr.Dataset:
    PHAMAT_TO_IDX = [
        # Coefficients for spherical particles
        ("11", 0),
        ("12", 1),
        ("33", 2),
        ("34", 3),
        # Additional for spheroidal particles
        ("22", 4),
        ("44", 5),
    ]

    # Gather wavelengths
    units = _get_units(ds, "wavelen", fallback_units)
    w = ds["wavelen"].values * units
    nw = len(w)

    # Gather extinction coefficients
    units = _get_units(ds, "ext", fallback_units)
    ext = ds["ext"].values * units

    # Gather albedo
    units = _get_units(ds, "ssa", fallback_units)
    ssa = ds["ssa"].values * units

    # Process phase function entries
    nangles = ds.sizes["nthetamax"]
    nphamat = ds.sizes["nphamat"]
    phamat = [x for x, _ in PHAMAT_TO_IDX[:nphamat]]

    # --- Allocate result arrays ---
    phase = np.full((nphamat, nw, nangles), np.nan)
    theta = np.full((nw, nangles), np.nan)

    # --- For each wavelength, refine the scattering angle grid
    #     and interpolate the phase function on it ---
    for iw in range(nw):
        # Refine scattering angle grid, interpolating nans linearly
        ntheta_ = int(ds["ntheta"].isel(nlam=iw, nphamat=0).values)
        theta_ = (
            ds["theta"].isel(nlam=iw, nphamat=0).values[:ntheta_]
        )  # keep only mesh for (1,1) component
        theta_refined = np.full((nangles,), np.nan)
        idx_dst = np.linspace(0, nangles - 1, ntheta_, dtype="int32")
        theta_refined[idx_dst] = theta_

        for i, j in zip(idx_dst[:-1], idx_dst[1:]):
            interpolated = np.linspace(
                theta_refined[i], theta_refined[j], j - i + 1, dtype=theta_refined.dtype
            )[1:-1]
            theta_refined[i + 1 : j] = interpolated

        # Sort input angles in ascending order
        idx_sort = np.argsort(theta_)
        phase_ = ds["phase"].isel(nlam=iw).values[:, idx_sort]
        theta_ = theta_[idx_sort]

        # Resample phase function using high-performance 1D interpolator with spectator dims support
        phase_refined = interp1d(theta_, phase_, theta_refined)

        # Store resampled angular grid and phase function
        theta[iw, :] = theta_refined
        phase[:, iw, :] = phase_refined

    # TODO: Check if interpolation in mu space wouldn't be better
    mu = np.cos(np.deg2rad(theta))
    # TODO: Check if normalization step is needed
    phase *= ureg("1 / sr")
    theta *= ureg("deg")
    attrs = {}

    return make_aer_core_v2(
        w=w,
        phamat=phamat,
        mu=mu,
        theta=theta,
        ext=ext,
        ssa=ssa,
        phase=phase,
        attrs=attrs,
    )


ds = xr.open_dataset("wc.sol.mie.cdf").sel(nreff=0)
# display(ds)
libradtran_to_aer_core_v2(ds, fallback_units={"ssa": "dimensionless"})

In [ ]:
import drjit as dr
import mitsuba as mi

mi.set_variant("scalar_mono_double")

mus_a = np.linspace(-1, 1, 11)
values_a = np.linspace(0, 1, 11)
mus_b = np.linspace(-1, 1, 21)
values_b = np.ones_like(mus_b)

mus_init = np.linspace(-1, 1, np.max((len(mus_a), len(mus_b))))
values_init = np.ones_like(mus_init)

# Bug dans tabphase_polarized ? Est-ce que les valeurs des nœuds de la distribution
# sont bien mises à jour quand on fait une update ?
p = mi.load_dict(
    {
        "type": "tabphase_polarized",
        "m11": ",".join(map(str, values_init)),
        "nodes": ",".join(map(str, mus_init)),
    }
)
params = mi.traverse(p)

values_update = np.zeros_like(values_init)
values_update[: len(mus_a)] = values_a
mus_update = np.zeros_like(mus_init)
mus_update[: len(mus_a)] = mus_a
mus_update[len(mus_a) :] = np.linspace(1.00001, 2, len(mus_init) - len(mus_a))

params["nodes"] = mus_update
params["m11"] = values_update
params.update()
print(p)

ctx = dr.zeros(mi.PhaseFunctionContext)
mei = mi.MediumInteraction3f()
mei.wi = (0, 0, 1)
wo = mi.Vector3f(0, 0, 1)
p.eval_pdf(ctx, mei, wo)